#Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:Llama-3.3-70b-versatile")

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year movie was released")
    director: str=Field(description="The person who directed the movie")
    rating: float=Field(description="Rating of the movie out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("Provide details about Avenger End Game")

Movie(title='Avenger End Game', year=2019, director='Anthony Russo, Joseph Russo', rating=8.4)

#Nested Structure

In [4]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year movie was released")
    director: str=Field(description="The person who directed the movie")
    rating: float=Field(description="Rating of the movie out of 10")
    cast: list[Actor]
    genres: list[str]
    budget: float | None=Field(description="Budget of the movie in millions USD")

model_with_structured= model.with_structured_output(MovieDetails)

model_with_structured.invoke("Provide details about The Amazing Spiderman")

MovieDetails(title='The Amazing Spider-Man', year=2012, director='Marc Webb', rating=7.4, cast=[Actor(name='Andrew Garfield', role='Spider-Man / Peter Parker'), Actor(name='Emma Stone', role='Gwen Stacy')], genres=['Action', 'Adventure', 'Thriller'], budget=230.0)

#Typed Dict- Provides a simpler alternative for structured output using Python's built-in typing, ideal when the runtime validation is not required

In [5]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"The year movie was released"]
    director: Annotated[str,...,"The person who directed the movie"]
    rating: Annotated[int,...,"Rating of the movie out of 10"]

model_with_typed_dict=model.with_structured_output(MovieDict)
model_with_typed_dict.invoke("Provide details about Iron Man")

{'director': 'Jon Favreau', 'rating': 8, 'title': 'Iron Man', 'year': 2008}

In [7]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    director: str
    rating: float
    cast: list[Actor]
    genres: list[str]
    budget: float | None

model_with_structured= model.with_structured_output(MovieDetails)

model_with_structured.invoke("Provide details about Conjuring")

{'budget': 20,
 'cast': [{'name': 'Vera Farmiga', 'role': 'Lorraine Warren'},
  {'name': 'Patrick Wilson', 'role': 'Ed Warren'}],
 'director': 'James Wan',
 'genres': ['Horror', 'Mystery', 'Thriller'],
 'rating': 7.9,
 'title': 'The Conjuring',
 'year': 2013}

#Data Classes- It is typically a class containing mainly data, although there arent any restrictions. We can create it using @dataclass decorator

In [18]:
import os
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

class ContactInfo(BaseModel):
    name: str = Field(description="Name of the person")
    email: str = Field(description="Email of the person")
    phone: str = Field(description="Phone Number of the person")

agent = create_agent(
    model = "google_genai:gemini-2.5-flash-lite",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john.doe@example.com, (555) 123-4567"
        }
    ]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john.doe@example.com', phone='(555) 123-4567')

In [9]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "google_genai:gemini-2.5-flash-lite",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john.doe@example.com, (555) 123-4567"
        }
    ]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john.doe@example.com', phone='(555) 123-4567')